# Interactive KC-string optimisation

Run **all cells once** to create the six panels, then use their buttons in order.
Choose this project's Pixi Python kernel in VS Code/Jupyter. Run `pixi install`
after updating the repository. A live kernel is required for controls and audio.

The simulator uses the native first-order forward-mode implementation. The first
synthesis may compile the C++ extension. Equations and implementation details
remain in `kc_string.ipynb` and `src/kc_string_forward.py`.

All **eight supported parameters** are editable and selectable for optimisation.
The excitation remains the reference's **first sine mode**, and the listener
remains at the **midpoint**; this model has no pluck/listener position controls.


In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import torch
from IPython.display import display

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "src" / "kc_string_forward.py").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.optimization_workbench import OptimizationWorkbench

torch.set_num_threads(1)
workbench = OptimizationWorkbench()


## 1. Choose parameters and synthesise the target

Enter **physical values**, not multipliers. Tick the parameters you want to fit;
all eight are initially selected. Unticked parameters will stay at their **initial
signal's values**, not their target values.

Set audio sampling rate, duration, spatial intervals and fixed oversampling.
Both sounds and both optimisers use this same configuration. With oversampling,
the native string runs at `audio rate × oversampling`, followed by a differentiable
windowed-sinc low-pass filter and decimation. At factor one there is no resampling.

Click **Generate target**. Invalid values or a failing linear stability bound are
reported without changing your entries. The default CeL needs at least 1024 output
samples. The stability check is not a complete nonlinear stability guarantee.


In [ ]:
display(workbench.target_panel())

## 2. Choose parameters and synthesise the initial signal

Edit the same eight physical parameters independently. You can copy the target
or apply the previous experiment's signed 10% perturbation, then edit further.
Click **Generate initial signal** to hear it and see the starting CeL.

The target and initial signal share rate, duration and grid. Playback uses one
shared gain; the loss always receives the original, unnormalised signals.
Editing synthesis settings invalidates stale sounds and results. Editing only
the optimisation checkboxes preserves the sounds but clears optimisation results.


In [ ]:
display(workbench.initial_panel())

## 3. Run Adam

Choose its learning rate and maximum number of steps, then click **Run ADAM**.
It always starts from the generated initial signal. Only ticked parameters are
updated. Stable trials use the library's default
`STFTCumulativeEnergyLoss(sample_rate=audio_rate)`: FFT 1024, hop 256, periodic
Hann window, four diagonal directions, no log weighting or decay.

**Patience:** stop if the best loss improves by **less than 10% over the last
20 completed optimiser steps**. The maximum step count is a separate limit.
Positive quantities use log coordinates; shifted log coordinates allow zero
initial damping. Proposals violating positivity or the fixed-grid stability
bound are rejected. No extra regulariser is added to valid CeL evaluations.


In [ ]:
display(workbench.optimizer_panel("adam"))

## 4. Show and play Adam's prediction

This panel updates automatically when Adam finishes. It uses the **best-loss
checkpoint**, not necessarily the last step. Compare the parameter table, loss
curves and target/initial/predicted spectrograms. Spectrograms use a shared dB
scale and the loss's STFT settings, with a 0–2 kHz frequency view.


In [ ]:
display(workbench.prediction_panel("adam"))

## 5. Run L-BFGS from the same initial signal

L-BFGS starts independently from the same generated initial parameters—not from
Adam's answer—and uses the same selected parameters, target and loss. It uses a
strong-Wolfe line search and one internal iteration per displayed optimiser step.
Line-search evaluations are reported separately. The **same 20-step / 10% patience**
rule applies. Controls are locked while either optimiser is running.


In [ ]:
display(workbench.optimizer_panel("lbfgs"))

## 6. Show and play L-BFGS's prediction

Compare its best checkpoint with the target and initial signal. Once both runs
exist, the loss plots show both optimisers. A good signal fit does **not** guarantee
unique recovery of all physical parameters: for example, jointly scaling density,
tension and Young's modulus can preserve the model's ratios.

Results remain available as `workbench.results["adam"]` and
`workbench.results["lbfgs"]`. Each contains the best physical parameters, audio,
loss history, selected mask, timing and stopping reason. Target and initial
snapshots are in `workbench.target` and `workbench.initial`.


In [ ]:
display(workbench.prediction_panel("lbfgs"))